# 03. Preparación de Datos para Dashboard

## Objetivo

Preparar tablas analíticas y KPIs finales a partir de los datos limpios del proyecto Online Retail.

El objetivo de esta etapa es transformar los resultados del análisis exploratorio en estructuras listas para visualización y toma de decisiones.

Se prepararán datasets orientados a:

- KPIs generales del negocio.
- Evolución temporal de ventas.
- Rendimiento por país.
- Rendimiento por producto.
- Comportamiento de clientes.
- Segmentación RFM.
- Concentración de ingresos.
- Pedidos de alto volumen.

Estos archivos servirán como fuente para el dashboard final del proyecto.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Entorno de preparación para dashboard listo")

Entorno de preparación para dashboard listo


In [2]:
# Rutas del proyecto

clean_path = Path("data/clean")
dashboard_path = Path("data/dashboard")

# Crear carpeta de salida si no existe
dashboard_path.mkdir(parents=True, exist_ok=True)

print("Carpeta dashboard creada:", dashboard_path.exists())

Carpeta dashboard creada: True


In [3]:
# Cargar dataset de ventas válidas

sales = pd.read_csv(
    clean_path / "online_retail_valid_sales.csv",
    parse_dates=["InvoiceDate"],
    low_memory=False
)

print("Ventas cargadas correctamente")
print("Filas:", f"{len(sales):,}")
print("Columnas:", sales.shape[1])

sales.head()

Ventas cargadas correctamente
Filas: 524,878
Columnas: 19


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsNegativeQuantity,IsZeroPrice,IsNegativePrice,ValidSale,GrossLineValue,Year,Month,YearMonth,DayOfWeek,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom,False,False,False,False,True,15.30,2010,12,2010-12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,False,False,False,True,20.34,2010,12,2010-12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom,False,False,False,False,True,22.00,2010,12,2010-12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,False,False,False,True,20.34,2010,12,2010-12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom,False,False,False,False,True,20.34,2010,12,2010-12,Wednesday,8


In [4]:
# ============================================================
# KPIs EJECUTIVOS DEL NEGOCIO
# ============================================================

# Revenue total
total_revenue = sales["GrossLineValue"].sum()

# Número de pedidos únicos
total_orders = sales["InvoiceNo"].nunique()

# Clientes identificados
total_customers = sales["CustomerID"].nunique()

# Unidades vendidas
total_units = sales["Quantity"].sum()

# Ticket promedio por pedido
avg_order_value = total_revenue / total_orders

# Revenue promedio por cliente
avg_revenue_customer = total_revenue / total_customers

# Crear tabla de KPIs
executive_kpis = pd.DataFrame({
    "KPI": [
        "Revenue Total",
        "Pedidos",
        "Clientes Identificados",
        "Unidades Vendidas",
        "Ticket Promedio",
        "Revenue Promedio por Cliente"
    ],
    "Valor": [
        total_revenue,
        total_orders,
        total_customers,
        total_units,
        avg_order_value,
        avg_revenue_customer
    ]
})

executive_kpis

,KPI,Valor
0,Revenue Total,"10,642,110.80"
1,Pedidos,"19,960.00"
2,Clientes Identificados,"4,338.00"
3,Unidades Vendidas,"5,572,420.00"
4,Ticket Promedio,533.17
5,Revenue Promedio por Cliente,"2,453.23"


In [5]:
# ============================================================
# VALIDACIÓN DE REVENUE: IDENTIFICADO VS NO IDENTIFICADO
# ============================================================

identified_sales = sales[sales["CustomerID"].notna()]
unidentified_sales = sales[sales["CustomerID"].isna()]

identified_revenue = identified_sales["GrossLineValue"].sum()
unidentified_revenue = unidentified_sales["GrossLineValue"].sum()

identified_share = identified_revenue / total_revenue * 100
unidentified_share = unidentified_revenue / total_revenue * 100

revenue_validation = pd.DataFrame({
    "Tipo de cliente": [
        "Cliente identificado",
        "Cliente no identificado",
        "Total"
    ],
    "Revenue": [
        identified_revenue,
        unidentified_revenue,
        total_revenue
    ],
    "% del Revenue": [
        identified_share,
        unidentified_share,
        100
    ]
})

revenue_validation.round(2)

,Tipo de cliente,Revenue,% del Revenue
0,Cliente identificado,"8,887,208.89",83.51
1,Cliente no identificado,"1,754,901.91",16.49
2,Total,"10,642,110.80",100.00


In [6]:
# ============================================================
# EVOLUCIÓN MENSUAL DEL NEGOCIO
# ============================================================

# Crear copia para preparación del dashboard
dashboard_sales = sales.copy()

# Crear periodo mensual
dashboard_sales["Month"] = (
    dashboard_sales["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

# Agregación mensual
monthly_performance = (
    dashboard_sales
    .groupby("Month")
    .agg(
        Revenue=("GrossLineValue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity", "sum"),
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

# Ticket promedio mensual
monthly_performance["Avg_Order_Value"] = (
    monthly_performance["Revenue"] /
    monthly_performance["Orders"]
)

monthly_performance.round(2)

,Month,Revenue,Orders,Units,Customers,Avg_Order_Value
0,2010-12,"821,452.73",1559,358019,885,526.91
1,2011-01,"689,811.61",1086,387099,741,635.19
2,2011-02,"522,545.56",1100,282934,758,475.04
3,2011-03,"716,215.26",1454,376599,974,492.58
4,2011-04,"536,968.49",1246,307953,856,430.95
5,2011-05,"769,296.61",1681,395001,1056,457.64
6,2011-06,"760,547.01",1533,388511,991,496.12
7,2011-07,"718,076.12",1475,399693,949,486.83
8,2011-08,"757,841.38",1361,421020,935,556.83
9,2011-09,"1,056,435.19",1837,569573,1266,575.09


In [7]:
# ============================================================
# DESEMPEÑO POR PAÍS
# ============================================================

country_performance = (
    dashboard_sales
    .groupby("Country")
    .agg(
        Revenue=("GrossLineValue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Customers=("CustomerID", "nunique"),
        Units=("Quantity", "sum")
    )
    .reset_index()
)

# Ticket promedio
country_performance["Avg_Order_Value"] = (
    country_performance["Revenue"] /
    country_performance["Orders"]
)

# Participación sobre el revenue total
country_performance["Revenue_Share_%"] = (
    country_performance["Revenue"] /
    country_performance["Revenue"].sum()
    * 100
)

# Ordenar de mayor a menor revenue
country_performance = (
    country_performance
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
)

country_performance.head(15).round(2)

,Country,Revenue,Orders,Customers,Units,Avg_Order_Value,Revenue_Share_%
0,United Kingdom,"9,001,744.09",18019,3920,4646906,499.57,84.59
1,Netherlands,"285,446.34",94,9,200361,"3,036.66",2.68
2,EIRE,"283,140.52",288,3,147007,983.13,2.66
3,Germany,"228,678.40",457,94,119154,500.39,2.15
4,France,"209,625.37",392,87,112060,534.76,1.97
5,Australia,"138,453.81",57,9,83891,"2,429.01",1.30
6,Spain,"61,558.56",90,30,27933,683.98,0.58
7,Switzerland,"57,067.60",54,21,30617,"1,056.81",0.54
8,Belgium,"41,196.34",98,25,23237,420.37,0.39
9,Sweden,"38,367.83",36,8,36078,"1,065.77",0.36


In [8]:
# ============================================================
# DESEMPEÑO POR PRODUCTO
# ============================================================

product_performance = (
    dashboard_sales
    .groupby(["StockCode", "Description"])
    .agg(
        Revenue=("GrossLineValue", "sum"),
        Units=("Quantity", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

# Revenue promedio por pedido
product_performance["Revenue_Per_Order"] = (
    product_performance["Revenue"] /
    product_performance["Orders"]
)

# Participación sobre revenue total
product_performance["Revenue_Share_%"] = (
    product_performance["Revenue"] /
    product_performance["Revenue"].sum()
    * 100
)

# Ordenar por revenue
product_performance = (
    product_performance
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
)

product_performance.head(15).round(2)

,StockCode,Description,Revenue,Units,Orders,Customers,Revenue_Per_Order,Revenue_Share_%
0,DOT,DOTCOM POSTAGE,"206,248.77",706,706,1,292.14,1.94
1,22423,REGENCY CAKESTAND 3 TIER,"174,156.54",13851,1988,881,87.60,1.64
2,23843,"PAPER CRAFT , LITTLE BIRDIE","168,469.60",80995,1,1,"168,469.60",1.58
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"104,284.24",37580,2189,856,47.64,0.98
4,47566,PARTY BUNTING,"99,445.23",18283,1685,708,59.02,0.93
5,85099B,JUMBO BAG RED RETROSPOT,"94,159.81",48371,2089,635,45.07,0.88
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,"81,700.92",78033,247,138,330.77,0.77
7,POST,POSTAGE,"78,101.88",3150,1126,331,69.36,0.73
8,M,Manual,"77,750.27",6984,289,197,269.03,0.73
9,23084,RABBIT NIGHT LIGHT,"66,870.03",30739,994,450,67.27,0.63


In [9]:
# ============================================================
# TOP PRODUCTOS COMERCIALES
# Excluir cargos administrativos / operativos
# ============================================================

non_product_codes = [
    "DOT",
    "POST",
    "M"
]

commercial_products = (
    product_performance[
        ~product_performance["StockCode"].astype(str).isin(non_product_codes)
    ]
    .copy()
)

# Top 15 productos comerciales por revenue
top_commercial_products = (
    commercial_products
    .sort_values("Revenue", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

top_commercial_products.round(2)

,StockCode,Description,Revenue,Units,Orders,Customers,Revenue_Per_Order,Revenue_Share_%
0,22423,REGENCY CAKESTAND 3 TIER,"174,156.54",13851,1988,881,87.60,1.64
1,23843,"PAPER CRAFT , LITTLE BIRDIE","168,469.60",80995,1,1,"168,469.60",1.58
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"104,284.24",37580,2189,856,47.64,0.98
3,47566,PARTY BUNTING,"99,445.23",18283,1685,708,59.02,0.93
4,85099B,JUMBO BAG RED RETROSPOT,"94,159.81",48371,2089,635,45.07,0.88
5,23166,MEDIUM CERAMIC TOP STORAGE JAR,"81,700.92",78033,247,138,330.77,0.77
6,23084,RABBIT NIGHT LIGHT,"66,870.03",30739,994,450,67.27,0.63
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,"64,875.59",19329,1160,613,55.93,0.61
8,84879,ASSORTED COLOUR BIRD ORNAMENT,"58,927.62",36362,1455,678,40.50,0.55
9,79321,CHILLI LIGHTS,"54,096.36",10302,661,205,81.84,0.51


In [11]:
# ============================================================
# SEGMENTACIÓN RFM PARA DASHBOARD
# ============================================================

# Trabajar únicamente con clientes identificados
rfm_sales = dashboard_sales[
    dashboard_sales["CustomerID"].notna()
].copy()

# Fecha de referencia: un día después de la última compra
snapshot_date = (
    rfm_sales["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

# Construir métricas RFM
rfm_dashboard = (
    rfm_sales
    .groupby("CustomerID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (snapshot_date - x.max()).days
        ),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("GrossLineValue", "sum")
    )
    .reset_index()
)

print("Fecha de referencia:", snapshot_date)
print("Clientes RFM:", len(rfm_dashboard))

rfm_dashboard.head()

Fecha de referencia: 2011-12-10 12:50:00
Clientes RFM: 4338


,CustomerID,Recency,Frequency,Monetary
0,"12,346.00",326,1,"77,183.60"
1,"12,347.00",2,7,"4,310.00"
2,"12,348.00",75,4,"1,797.24"
3,"12,349.00",19,1,"1,757.55"
4,"12,350.00",310,1,334.40


In [12]:
# ============================================================
# SCORES RFM
# ============================================================

# Recency:
# Menor número de días = mejor comportamiento = mayor score
rfm_dashboard["R_Score"] = pd.qcut(
    rfm_dashboard["Recency"].rank(method="first"),
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency:
# Mayor frecuencia = mayor score
rfm_dashboard["F_Score"] = pd.qcut(
    rfm_dashboard["Frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary:
# Mayor valor monetario = mayor score
rfm_dashboard["M_Score"] = pd.qcut(
    rfm_dashboard["Monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Crear código RFM
rfm_dashboard["RFM_Score"] = (
    rfm_dashboard["R_Score"].astype(str)
    + rfm_dashboard["F_Score"].astype(str)
    + rfm_dashboard["M_Score"].astype(str)
)

print("Scores RFM creados correctamente")
print("Clientes:", len(rfm_dashboard))

rfm_dashboard.head(10)

Scores RFM creados correctamente
Clientes: 4338


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,"12,346.00",326,1,"77,183.60",1,1,5,115
1,"12,347.00",2,7,"4,310.00",5,5,5,555
2,"12,348.00",75,4,"1,797.24",2,4,4,244
3,"12,349.00",19,1,"1,757.55",4,1,4,414
4,"12,350.00",310,1,334.40,1,1,2,112
5,"12,352.00",36,8,"2,506.04",3,5,5,355
6,"12,353.00",204,1,89.00,1,1,1,111
7,"12,354.00",232,1,"1,079.40",1,1,4,114
8,"12,355.00",214,1,459.40,1,1,2,112
9,"12,356.00",23,3,"2,811.43",4,3,5,435


In [13]:
# ============================================================
# SEGMENTACIÓN RFM
# Misma metodología utilizada en 02_exploratory_analysis
# ============================================================

def assign_rfm_segment(row):
    r = row["R_Score"]
    f = row["F_Score"]

    if r >= 4 and f >= 4:
        return "Champions"

    elif r >= 3 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f in [2, 3]:
        return "Potential Loyalists"

    elif r == 5 and f == 1:
        return "New Customers"

    elif r == 3 and f in [2, 3]:
        return "Needs Attention"

    elif r <= 2 and f >= 4:
        return "At Risk"

    elif r <= 2 and f in [2, 3]:
        return "Hibernating"

    elif r <= 2 and f == 1:
        return "Lost"

    else:
        return "Other"


# Aplicar segmentación
rfm_dashboard["Segment"] = rfm_dashboard.apply(
    assign_rfm_segment,
    axis=1
)

# Crear tabla de validación
segment_validation = (
    rfm_dashboard["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="Customers")
)

segment_validation["Customer_Share_%"] = (
    segment_validation["Customers"]
    / len(rfm_dashboard)
    * 100
)

print("Segmentación RFM completada")
print("Total de clientes:", segment_validation["Customers"].sum())

segment_validation.round(2)

Segmentación RFM completada
Total de clientes: 4338


,Segment,Customers,Customer_Share_%
0,Champions,1121,25.84
1,Hibernating,885,20.40
2,Lost,565,13.02
3,Potential Loyalists,477,11.00
4,Needs Attention,373,8.60
5,Loyal Customers,329,7.58
6,At Risk,285,6.57
7,Other,261,6.02
8,New Customers,42,0.97


In [14]:
# ============================================================
# RENDIMIENTO ECONÓMICO POR SEGMENTO RFM
# ============================================================

segment_performance_dashboard = (
    rfm_dashboard
    .groupby("Segment")
    .agg(
        Customers=("CustomerID", "nunique"),
        Revenue=("Monetary", "sum"),
        Avg_Revenue_Per_Customer=("Monetary", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Recency=("Recency", "mean")
    )
    .reset_index()
)

# Participación de cada segmento en el revenue identificado
segment_performance_dashboard["Revenue_Share_%"] = (
    segment_performance_dashboard["Revenue"]
    / segment_performance_dashboard["Revenue"].sum()
    * 100
)

# Ordenar por contribución económica
segment_performance_dashboard = (
    segment_performance_dashboard
    .sort_values("Revenue", ascending=False)
    .reset_index(drop=True)
)

print("Tabla de rendimiento RFM creada correctamente")
print("Clientes:", segment_performance_dashboard["Customers"].sum())
print(
    "Revenue identificado:",
    round(segment_performance_dashboard["Revenue"].sum(), 2)
)

segment_performance_dashboard.round(2)

Tabla de rendimiento RFM creada correctamente
Clientes: 4338
Revenue identificado: 8887208.89


,Segment,Customers,Revenue,Avg_Revenue_Per_Customer,Avg_Frequency,Avg_Recency,Revenue_Share_%
0,Champions,1121,"5,857,798.93","5,225.51",10.05,13.00,65.91
1,Loyal Customers,329,"807,776.94","2,455.25",5.72,48.97,9.09
2,Hibernating,885,"603,425.79",681.84,1.67,190.72,6.79
3,Potential Loyalists,477,"496,163.25","1,040.17",2.01,16.62,5.58
4,At Risk,285,"453,355.70","1,590.72",4.89,134.79,5.10
5,Lost,565,"290,934.23",514.93,1.00,221.39,3.27
6,Needs Attention,373,"258,243.89",692.34,1.82,51.54,2.91
7,Other,261,"103,339.22",395.94,1.00,42.22,1.16
8,New Customers,42,"16,170.94",385.02,1.00,6.86,0.18


In [15]:
# ============================================================
# VALIDACIÓN FINAL ANTES DE EXPORTAR
# ============================================================

print("=" * 60)
print("VALIDACIÓN FINAL DE DATOS PARA DASHBOARD")
print("=" * 60)

# 1. Revenue
print("\n1. REVENUE")
print(f"Revenue total: {total_revenue:,.2f}")
print(f"Revenue identificado: {identified_revenue:,.2f}")
print(f"Revenue no identificado: {unidentified_revenue:,.2f}")

# 2. Clientes
print("\n2. CLIENTES")
print(f"Clientes identificados: {total_customers:,}")
print(f"Clientes RFM: {len(rfm_dashboard):,}")

# 3. Pedidos
print("\n3. PEDIDOS")
print(f"Pedidos únicos: {total_orders:,}")

# 4. Segmentación
print("\n4. SEGMENTACIÓN RFM")
print(
    "Clientes distribuidos en segmentos:",
    segment_performance_dashboard["Customers"].sum()
)

print(
    "Revenue RFM:",
    round(segment_performance_dashboard["Revenue"].sum(), 2)
)

# 5. Comprobaciones
print("\n5. COMPROBACIONES")

print(
    "Clientes KPI = Clientes RFM:",
    total_customers == len(rfm_dashboard)
)

print(
    "Clientes RFM = Clientes segmentados:",
    len(rfm_dashboard)
    == segment_performance_dashboard["Customers"].sum()
)

print(
    "Revenue identificado = Revenue RFM:",
    round(identified_revenue, 2)
    == round(segment_performance_dashboard["Revenue"].sum(), 2)
)

print("\n" + "=" * 60)
print("VALIDACIÓN TERMINADA")
print("=" * 60)

VALIDACIÓN FINAL DE DATOS PARA DASHBOARD

1. REVENUE
Revenue total: 10,642,110.80
Revenue identificado: 8,887,208.89
Revenue no identificado: 1,754,901.91

2. CLIENTES
Clientes identificados: 4,338
Clientes RFM: 4,338

3. PEDIDOS
Pedidos únicos: 19,960

4. SEGMENTACIÓN RFM
Clientes distribuidos en segmentos: 4338
Revenue RFM: 8887208.89

5. COMPROBACIONES
Clientes KPI = Clientes RFM: True
Clientes RFM = Clientes segmentados: True
Revenue identificado = Revenue RFM: True

VALIDACIÓN TERMINADA


In [16]:
# ============================================================
# EXPORTAR DATASETS FINALES PARA DASHBOARD
# ============================================================

from pathlib import Path

# Crear carpeta de exportación
dashboard_path = Path("data/dashboard")
dashboard_path.mkdir(parents=True, exist_ok=True)

# 1. KPIs ejecutivos
executive_kpis.to_csv(
    dashboard_path / "executive_kpis.csv",
    index=False
)

# 2. Rendimiento mensual
monthly_performance.to_csv(
    dashboard_path / "monthly_performance.csv",
    index=False
)

# 3. Rendimiento por país
country_performance.to_csv(
    dashboard_path / "country_performance.csv",
    index=False
)

# 4. Rendimiento de productos
top_commercial_products.to_csv(
    dashboard_path / "product_performance.csv",
    index=False
)

# 5. Clientes RFM
rfm_dashboard.to_csv(
    dashboard_path / "customer_rfm.csv",
    index=False
)

# 6. Rendimiento por segmento RFM
segment_performance_dashboard.to_csv(
    dashboard_path / "rfm_segment_performance.csv",
    index=False
)

# 7. Validación de revenue identificado/no identificado
revenue_validation.to_csv(
    dashboard_path / "revenue_validation.csv",
    index=False
)

print("=" * 60)
print("EXPORTACIÓN PARA DASHBOARD COMPLETADA")
print("=" * 60)

for archivo in sorted(dashboard_path.glob("*.csv")):
    print(f"✓ {archivo.name}")

print("\nCarpeta:", dashboard_path.resolve())

EXPORTACIÓN PARA DASHBOARD COMPLETADA
✓ country_performance.csv
✓ customer_rfm.csv
✓ executive_kpis.csv
✓ monthly_performance.csv
✓ product_performance.csv
✓ revenue_validation.csv
✓ rfm_segment_performance.csv

Carpeta: /Users/christopherlunagonzalez/Documents/Data_Analyst_Portfolio/Project_01_Online_Retail/data/dashboard


## Dashboard interactivo — Tableau Public

Como etapa final del proyecto, se desarrolló un dashboard interactivo en Tableau Public para presentar los principales resultados del análisis de ventas y clientes.

El dashboard incluye:

- Resumen ejecutivo de ventas y KPIs principales.
- Tendencia mensual de ingresos.
- Análisis de ingresos por país y producto.
- Segmentación de clientes mediante RFM.
- Identificación de clientes Champions y clientes en riesgo.
- Análisis del valor generado por cada segmento de clientes.

### Dashboard en Tableau Public

https://public.tableau.com/views/E-CommerceSalesCustomerAnalytics_17868365296540/ExecutiveSalesOverview?:language=es-ES&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link